
This notebook takes BT-Settl models downloaded from http://svo2.cab.inta-csic.es/theory/newov2/index.php?models=bt-settl and does a few things with them:

(1) Convert from air wavelengths to vacuum wavelengths

(2) Resamples the models onto a standard wavelength grid, using a flux conserving algorithm (Spectres)

(3) Uses bilinear interpolation to place them on the same (logt, logg) grid used by the other stellar libraries in FSPS.

(4) Converts the units from per-unit wavelength to per-unit frequency [erg/s/cm2/angstrom -> erg/s/cm2/Hz/sr]
    
(5) Converts them into the same binary format that FSPS uses for all of its other stellar libraries.

In [156]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate
import tqdm
import re
import os
import glob
import spectres
from astropy import units as u

%matplotlib inline

In [157]:
SPS_HOME = os.path.abspath(os.path.join(os.getcwd(), '..'))
# SPS_HOME = os.getenv('SPS_HOME')
# SPS_HOME = SPS_HOME.replace('fsps', 'fsps_dev')  # -> I call my development folder for fsps 'fsps_dev' to keep it separate from my working fsps installation

# choose one metallicity value for this run
logzi = 0.5

print(SPS_HOME)

/Users/mreefe/Dropbox/Astrophysics/fsps_dev


In [158]:
# Define the teff, logg, and logz arrays that cover the grid of BT-Settl models

teff_1 = np.arange(400, 7000, 100)
teff_2 = np.arange(7000, 12000, 200)
teff_3 = np.arange(12000, 20000, 500)
teff_4 = np.arange(20000, 71000, 1000)
teff = np.concatenate((teff_1, teff_2, teff_3, teff_4))
logt = np.log10(teff)

logg = np.arange(-0.5, 6.5, 0.5)

logz = np.arange(-4.0, 0.5, 0.5)
logz = np.concatenate((logz, [0.3, 0.5]))

print(logt)
print(logg)
print(logz)

[2.60205999 2.69897    2.77815125 2.84509804 2.90308999 2.95424251
 3.         3.04139269 3.07918125 3.11394335 3.14612804 3.17609126
 3.20411998 3.23044892 3.25527251 3.2787536  3.30103    3.32221929
 3.34242268 3.36172784 3.38021124 3.39794001 3.41497335 3.43136376
 3.44715803 3.462398   3.47712125 3.49136169 3.50514998 3.51851394
 3.53147892 3.54406804 3.5563025  3.56820172 3.5797836  3.59106461
 3.60205999 3.61278386 3.62324929 3.63346846 3.64345268 3.65321251
 3.66275783 3.67209786 3.68124124 3.69019608 3.69897    3.70757018
 3.71600334 3.72427587 3.73239376 3.74036269 3.74818803 3.75587486
 3.76342799 3.77085201 3.77815125 3.78532984 3.79239169 3.79934055
 3.80617997 3.81291336 3.81954394 3.8260748  3.83250891 3.83884909
 3.84509804 3.8573325  3.86923172 3.88081359 3.8920946  3.90308999
 3.91381385 3.92427929 3.93449845 3.94448267 3.95424251 3.96378783
 3.97312785 3.98227123 3.99122608 4.         4.00860017 4.01703334
 4.02530587 4.03342376 4.04139269 4.04921802 4.05690485 4.0644

In [159]:
# Define the grid of teff and logg that cover the grid of FSPS's models
logt_fsps = np.loadtxt(os.path.join(SPS_HOME, 'SPECTRA/BaSeL3.1/basel_logt.dat'))
logg_fsps = np.loadtxt(os.path.join(SPS_HOME, 'SPECTRA/BaSeL3.1/basel_logg.dat'))

print(logt_fsps)
print(logg_fsps)

[3.30103 3.34242 3.39794 3.44716 3.47712 3.50515 3.52504 3.54407 3.57403
 3.60206 3.62839 3.65321 3.67669 3.69897 3.72016 3.74036 3.75967 3.77815
 3.79588 3.81291 3.8293  3.8451  3.86034 3.87506 3.8893  3.90309 3.91645
 3.92942 3.94201 3.95424 3.96614 3.97772 3.989   4.      4.02119 4.04139
 4.0607  4.07918 4.09691 4.11394 4.14613 4.17609 4.20412 4.23045 4.25527
 4.27875 4.30103 4.32222 4.34242 4.36173 4.38021 4.39794 4.41497 4.43136
 4.44716 4.4624  4.47712 4.49136 4.50515 4.51851 4.53148 4.54407 4.57403
 4.60206 4.62839 4.65321 4.67669 4.69897]
[-1.02 -0.7  -0.51 -0.29  0.    0.28  0.5   0.6   0.87  1.    1.5   2.
  2.5   3.    3.5   4.    4.5   5.    5.5 ]


In [160]:
# Define the wavelength grid that we will resample onto
# -> even logarithmic spacing by ~5% of the current wavelength
w_1 = 90. * 1.05**np.arange(0, int(np.log(800/90)/np.log(1.05))) 
# -> finer sampling in the FUV;
w_2 = np.arange(800., 1800., 0.2)
w_3 = np.arange(1800., 9000., 20.)
# -> spacing by ~5% of the current wavelength
w_4 = 9000. * 1.05**np.arange(0, int(np.log(9.99e6/9000)/np.log(1.05))) 

wavelength = np.concatenate((w_1, w_2, w_3, w_4))
print('w_1 = ', len(w_1))
print('w_2 = ', len(w_2))
print('w_3 = ', len(w_3))
print('w_4 = ', len(w_4))
print('length = ', len(wavelength))
print(wavelength)

w_1 =  44
w_2 =  5000
w_3 =  360
w_4 =  143
length =  5547
[9.00000000e+01 9.45000000e+01 9.92250000e+01 ... 8.33190634e+06
 8.74850165e+06 9.18592674e+06]


In [161]:
# wavelength must be in angstroms!
def airtovac(wavelength):
    # see: https://www.astro.uu.se/valdwiki/Air-to-vacuum%20conversion
    s = 1e4 / wavelength
    n = 1 + 0.00008336624212083 + 0.02408926869968 / (130.1065924522 - s**2) + 0.0001599740894897 / (38.92568793293 - s**2)
    # do not alter wavelengths below 2000 angstroms 
    wh = np.where(wavelength < 2000.)[0]
    n[wh] = 1.0
    return wavelength * n

In [162]:

# allocate a buffer for all of the spectra at one metallicity
btsettl_in = np.zeros((len(wavelength), len(logt), len(logg)))

folder = os.path.join(SPS_HOME, 'SPECTRA/BT-Settl/spectra')
files = glob.glob(os.path.join(folder, '*.txt'))

done = np.full(len(files), fill_value='', dtype='<U200')

i = 0
for fpath in tqdm.tqdm(files):

    # check if the file has already been processed - if so, skip it
    if fpath in done:
        continue

    # parse the file name to get the temp, logg, and logz
    fname = os.path.basename(fpath)
    m = re.search(r'^lte([0-9]+)([-+]\d\.\d)([-+]\d\.\d)a?([-+]\d\.\d)?\.BT-.*?\.txt$', fname)
    teff_v = int(m.group(1)) * 100
    logg_v = -float(m.group(2))
    logz_v = float(m.group(3))
    if m.group(4) is not None:
        alpha_v = float(m.group(4))
    else:
        alpha_v = 0.0
    assert logz_v == logzi, f"ERROR: Encountered a file with metallicity {logz_v} != {logzi}. Please clean the 'spectra' directory."

    # find the indices corresponding to these values in the array
    logt_i = np.where(teff == teff_v)[0][0]
    logg_i = np.where(logg == logg_v)[0][0]
    # print(f'Teff = {teff_v:.0f} (index = {logt_i:.0f})')
    # print(f'logg = {logg_v:.1f} (index = {logg_i:.0f})')
    # print(f'logZ = {logz_v:.1f}')

    # check for other models with different alpha enhancements
    fmatches = glob.glob(os.path.join(os.path.dirname(fpath), fname.replace(f'a{m.group(4)}', 'a*')))
    btsettl_o = np.zeros((len(wavelength), len(fmatches)))
    alpha_vs = np.zeros(len(fmatches))

    for j, fmatch in enumerate(fmatches):
        fname2 = os.path.basename(fmatch)
        m = re.search(r'^lte([0-9]+)([-+]\d\.\d)([-+]\d\.\d)a?([-+]\d\.\d)?\.BT-.*?\.txt$', fname2)
        if m.group(4) is not None:
            alpha_vs[j] = float(m.group(4))

        # read in the text file
        wave_i, btsettl_i = np.loadtxt(fmatch, unpack=True)

        # mask out the first wavelength point at 0 angstroms since it breaks to airtovac conversion
        wave_i = wave_i[1:]
        btsettl_i = btsettl_i[1:]

        # convert to vacuum wavelengths when lambda > 2000 angstroms
        wave_vac_i = airtovac(wave_i)

        # perform the flux-conserving resampling onto the output wavelength grid
        btsettl_o[:,j] = spectres.spectres(wavelength, wave_vac_i, btsettl_i, fill=0.)

        # make sure we dont redo this file later
        done[i] = fmatch
        i += 1

    # take the minimum (absolute val) alpha enhancement, since "most" stars are around 0
    minalpha = np.argmin(np.abs(alpha_vs))
    btsettl_o = btsettl_o[:,minalpha]

    # insert it into the 3D array
    btsettl_in[:, logt_i, logg_i] = btsettl_o

    # plot the new and old spectrum to compare them
    # fig, ax = plt.subplots()
    # ax.plot(wave_i, btsettl_i)
    # ax.plot(wavelength, btsettl_o)
    # ax.set_xscale('log')
    # ax.set_yscale('log')
    # ax.set_xlabel('Wavelength (angstroms)')
    # ax.set_ylabel('Flambda')
    # plt.show()
    # plt.close()


100%|██████████| 1302/1302 [04:46<00:00,  4.54it/s]


In [163]:
logg[0] = -0.51  # (this is just so that the interpolation doesnt think the -0.51 FSPS grid point is "out of bounds")

# Flatten the BT-Settl grid data
logt_flat, logg_flat = np.meshgrid(logt, logg, indexing='ij')
logt_flat = logt_flat.ravel()
logg_flat = logg_flat.ravel()
btsettl_flat = btsettl_in.reshape(len(wavelength), len(logt)*len(logg))

w5000 = np.nanargmin(np.abs(wavelength - 5000.))
good = btsettl_flat[w5000,:] > 0.

logt_flat = logt_flat[good]
logg_flat = logg_flat[good]
btsettl_flat = btsettl_flat[:,good]
grid_in = list(zip(logt_flat, logg_flat))

print(logt_flat.shape)
print(logg_flat.shape)
print(btsettl_flat.shape)

logt_fsps_flat, logg_fsps_flat = np.meshgrid(logt_fsps, logg_fsps, indexing='ij')
logt_fsps_flat = logt_fsps_flat.ravel()
logg_fsps_flat = logg_fsps_flat.ravel()

logt_fsps_coord, logg_fsps_coord = np.meshgrid(np.arange(len(logt_fsps)), np.arange(len(logg_fsps)), indexing='ij')
logt_fsps_coord = logt_fsps_coord.ravel()
logg_fsps_coord = logg_fsps_coord.ravel()

# decide which output grid points are "safe" to interpolate to (i.e. they dont fall outside the parameter space covered by the input grids)
good = np.zeros(len(logt_fsps_flat), dtype=bool)
for i in range(len(good)):

    lgi = logg_fsps_flat[i]
    lti = logt_fsps_flat[i]

    # if logt or logg is out of the whole range
    if lgi < logg.min() or lgi > logg.max() or lti < logt.min() or lti > logt.max():
        # pixel is bad - dont change the value in "good"
        continue

    # get the index of the closest temperature grid point 
    tind = np.nanargmin(np.abs(lti - logt))
    wht = np.where(logt_flat == logt[tind])[0]
    if len(wht) == 0:
        continue

    # get the range of logg values at these temperature grid points (assuming there are any)
    gmin = np.min(logg_flat[wht])
    gmax = np.max(logg_flat[wht])

    # finally, check if the logg for this point is within the bounds, and only then do
    # we allow this point to be interpolated to
    if lgi >= gmin and lgi <= gmax:
        good[i] = True

grid_out = list(zip(logt_fsps_flat[good], logg_fsps_flat[good]))
grid_out_coord = list(zip(logt_fsps_coord[good], logg_fsps_coord[good]))

print(len(grid_out))
print(len(logg_fsps)*len(logt_fsps))
print(grid_out)
print(grid_out_coord)

(1164,)
(1164,)
(5547, 1164)
731
1292
[(3.44716, -0.51), (3.44716, -0.29), (3.44716, 0.0), (3.44716, 0.28), (3.44716, 0.5), (3.44716, 0.6), (3.44716, 0.87), (3.44716, 1.0), (3.44716, 1.5), (3.44716, 2.0), (3.44716, 2.5), (3.44716, 3.0), (3.44716, 3.5), (3.44716, 4.0), (3.44716, 4.5), (3.44716, 5.0), (3.44716, 5.5), (3.47712, -0.51), (3.47712, -0.29), (3.47712, 0.0), (3.47712, 0.28), (3.47712, 0.5), (3.47712, 0.6), (3.47712, 0.87), (3.47712, 1.0), (3.47712, 1.5), (3.47712, 2.0), (3.47712, 2.5), (3.47712, 3.0), (3.47712, 3.5), (3.47712, 4.0), (3.47712, 4.5), (3.47712, 5.0), (3.47712, 5.5), (3.50515, -0.51), (3.50515, -0.29), (3.50515, 0.0), (3.50515, 0.28), (3.50515, 0.5), (3.50515, 0.6), (3.50515, 0.87), (3.50515, 1.0), (3.50515, 1.5), (3.50515, 2.0), (3.50515, 2.5), (3.50515, 3.0), (3.50515, 3.5), (3.50515, 4.0), (3.50515, 4.5), (3.50515, 5.0), (3.50515, 5.5), (3.52504, -0.51), (3.52504, -0.29), (3.52504, 0.0), (3.52504, 0.28), (3.52504, 0.5), (3.52504, 0.6), (3.52504, 0.87), (3.52504,

In [164]:

# allocate an output array buffer
btsettl_out = np.zeros((len(wavelength), len(logt_fsps), len(logg_fsps)))

# Do a bilinear interpolation across the logt and logg axes
for wi in tqdm.trange(len(wavelength)):

    # create the interpolator
    # interp = interpolate.RegularGridInterpolator((logt, logg), btsettl_in[wi,:,:], method='linear',
    #                                              bounds_error=False, fill_value=0.)
    # # interp = interpolate.LinearNDInterpolator(list(zip(logt_flat, logg_flat)), btsettl_flat[wi,:])
    # # create the meshgrid to interpolate onto
    # LOGT, LOGG = np.meshgrid(logt_fsps, logg_fsps, indexing='ij')
    # btsettl_out[wi,:,:] = interp((LOGT, LOGG))

    btout = interpolate.griddata(grid_in, btsettl_flat[wi, :], grid_out, method='linear', fill_value=0.)
    for k, gridpoint in enumerate(grid_out_coord):
        btsettl_out[wi, gridpoint[0], gridpoint[1]] = btout[k]


100%|██████████| 5547/5547 [05:00<00:00, 18.45it/s]


In [165]:
# Convert the units
c_ang = 299792458e10

for i in range(len(logt_fsps)):
    for j in range(len(logg_fsps)):
        btsettl_out[:,i,j] *= wavelength**2 / c_ang    # <= convert to erg/s/cm2/Hz
        btsettl_out[:,i,j] *= 1/(4*np.pi)              # <= convert to Harvard flux
        # note: insofar as I can tell, FSPS stores its stellar libraries in flux moment or "Harvard flux" units, 
        #       which are off from physical flux units by a factor of 4pi.  See page 244-245 in 
        #       https://ads.harvard.edu/books/1989fsa..book/AbookC09.pdf for more info.
        #       Also see line 199 in getspec.f90 which does the conversion from these units into Lsun/Hz.


In [166]:
# w_miles = np.loadtxt(os.path.join(SPS_HOME, 'SPECTRA/MILES/miles.lambda'))
# miles = np.fromfile(os.path.join(SPS_HOME, f'SPECTRA/MILES/imiles_z0.0008.spectra.bin'), dtype=np.float32)
# miles = miles.reshape(len(logg_fsps), len(logt_fsps), len(w_miles))

# template_folder = os.path.join(SPS_HOME, 'SPECTRA/BT-Settl/template_plots')
# if not os.path.exists(template_folder):
#     os.mkdir(template_folder)

# for j in range(len(logg_fsps)):
#     for i in range(len(logt_fsps)):
#         fig, ax = plt.subplots()
#         ax.plot(wavelength, btsettl_out[:,i,j], label='BT-Settl')
#         ax.plot(w_miles, miles[j,i,:], label='MILES+BaSeL')
#         ax.set_xscale('log')
#         ax.set_yscale('log')
#         ax.set_xlabel(r'Wavelength ($\mathring{\rm \AA}}$)')
#         ax.set_ylabel(r'Flux moment (erg s$^{-1}$ cm$^{-2}$ Hz$^{-1}$)')
#         ax.set_ylim(1e-42, 1e-2)
#         ax.set_title(f'logg={logg_fsps[j]}, teff={10**logt_fsps[i]}')
#         plt.savefig(os.path.join(template_folder, f'{j}_{i}.pdf'), dpi=300, bbox_inches='tight')
#         plt.close()

In [167]:
# Save as a binary file readable from fortran
# note: np.ndarray.tofile can only save in C-order (row-major) whereas fortran is column-major.
#       to get around this, we can just reverse the order of the axes before saving.
# note: BT-Settl grids use solar metallicity and abundances from Asplund et al. (2009), where Z = 0.0134
zsun = 0.0134
z = 10**logzi * zsun
btsettl_out.astype(np.float32).T.tofile(os.path.join(SPS_HOME, f'SPECTRA/BT-Settl/btsettl_z{z:.4f}.spectra.bin'))

In [168]:
# Store spectral resolution as -0 km/s
# Why? BT-Settl are atmospheric models so they don't really have a spectral resolution. The reason for the negative is to be consistent
# with FSPS's standard of using negative values for "poorly defined" spectral resolutions.
res = np.full((len(wavelength),), fill_value=-0.0)

In [169]:
# save other ancillary files
np.savetxt(os.path.join(SPS_HOME, 'SPECTRA/BT-Settl/btsettl.lambda'), wavelength, fmt='%12.4f')
np.savetxt(os.path.join(SPS_HOME, 'SPECTRA/BT-Settl/zlegend.dat'), 10**logz[4:]*zsun, fmt='%6.4f')
np.savetxt(os.path.join(SPS_HOME, 'SPECTRA/BT-Settl/btsettl.res'), res, fmt='%.1f')